# Infinite Connections: Generator and Audit Demo

This notebook is the backend reproducibility companion for the project.
The website is the main interactive demo. This notebook shows the core local pipeline:

1. load the historical NYT reference set,
2. generate a small sample of new boards,
3. run validation and ambiguity checks,
4. inspect one strong board and one flagged board,
5. summarize the resulting metrics.

In [ ]:

from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "infinite_connections").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from infinite_connections.generator_v2 import GeneratorV2
from infinite_connections.history import nearest_reference
from infinite_connections.schema import normalize_word
from infinite_connections.solver import solve_puzzle
from infinite_connections.validator import score_puzzle, validate_puzzle

DATA_DIR = PROJECT_ROOT / "data"
HISTORY_PATH = DATA_DIR / "history" / "unified_reference.json"
DASHBOARD_PATH = DATA_DIR / "reports" / "dashboard.json"

print("Project files located.")
print(f"History file: {HISTORY_PATH.name}")
print(f"Dashboard file: {DASHBOARD_PATH.name}")


In [ ]:
def load_reference_records(path: Path) -> list[dict]:
    raw = json.loads(path.read_text(encoding="utf-8"))
    items = raw if isinstance(raw, list) else raw.get("references", [])
    references = []
    for item in items:
        references.append(
            {
                "id": item.get("id", "reference"),
                "title": item.get("title", "reference"),
                "words": sorted({normalize_word(str(word)) for word in item.get("words", [])}),
            }
        )
    return references

references = load_reference_records(HISTORY_PATH)
dashboard = json.loads(DASHBOARD_PATH.read_text(encoding="utf-8"))

summary_rows = [
    ("Historical reference boards", len(references)),
    ("Main generation batch", dashboard["summary"]["generated"]),
    ("Curated play bank", dashboard["summary"]["published"]),
    ("Historical exact overlaps", dashboard["summary"]["history_exact_overlap"]),
    ("Blind unique-match rate (curated)", dashboard["curated_bank"]["blind_solver"]["unique_match_rate"]),
    ("Structural classifier F1", dashboard["summary"]["classifier_test_f1"]),
    ("Structural classifier AUC", dashboard["summary"]["classifier_test_auc"]),
    ("Curated plausibility pass rate", dashboard["summary"]["classifier_pass_rate"]),
]

for label, value in summary_rows:
    print(f"{label:40} {value}")

## Generate a small local sample

The notebook uses the same local mixed generator family as the main project, but only on a small sample.
This keeps the notebook fast and makes it easy to inspect the full pipeline end to end.

In [ ]:
generator = GeneratorV2(
    mode="mixed",
    theme_first_probability=0.25,
    use_ollama=False,
    rewrite_categories_with_ollama=False,
)

sample_puzzles = generator.generate(count=12, seed=561)
print(f"Generated sample size: {len(sample_puzzles)}")
print([p.title for p in sample_puzzles[:5]])

In [ ]:
def evaluate_puzzle(puzzle):
    issues = validate_puzzle(puzzle)
    nearest = nearest_reference(puzzle, references)
    report = score_puzzle(puzzle, nearest_reference=nearest)
    solver = solve_puzzle(puzzle, max_solutions=25)
    return {
        "puzzle": puzzle,
        "issues": issues,
        "nearest": nearest,
        "report": report,
        "solver": solver,
    }

results = [evaluate_puzzle(puzzle) for puzzle in sample_puzzles]

rows = []
for item in results:
    puzzle = item["puzzle"]
    report = item["report"]
    solver = item["solver"]
    nearest = item["nearest"] or {}
    rows.append(
        {
            "id": puzzle.id,
            "title": puzzle.title,
            "strategy": puzzle.source_strategy,
            "status": report.status,
            "quality_score": round(report.quality_score, 2),
            "solver_status": solver.status,
            "issue_count": len(item["issues"]),
            "nearest_similarity": round(float(nearest.get("similarity", 0.0)), 3),
        }
    )

try:
    import pandas as pd
    display(pd.DataFrame(rows).sort_values(["status", "quality_score"], ascending=[True, False]))
except Exception:
    for row in rows:
        print(row)

In [ ]:
status_counts = Counter(item["report"].status for item in results)
solver_counts = Counter(item["solver"].status for item in results)

sample_summary = {
    "sample_size": len(results),
    "publish": status_counts.get("publish", 0),
    "revise": status_counts.get("revise", 0),
    "reject": status_counts.get("reject", 0),
    "unique_match": solver_counts.get("unique_match", 0),
    "non_unique_or_flagged": len(results) - solver_counts.get("unique_match", 0),
    "mean_quality_score": round(sum(item["report"].quality_score for item in results) / len(results), 2),
}

sample_summary

## Inspect one strong board and one flagged board

I believe that a good demo should show both sides of the pipeline: a board that clears the screen and a board that is held back for revision.

In [ ]:

def board_markdown(item) -> str:
    puzzle = item["puzzle"]
    report = item["report"]
    solver = item["solver"]
    nearest = item["nearest"] or {}
    lines = []
    lines.append(f"### {puzzle.title}  ")
    lines.append(f"**Puzzle ID:** `{puzzle.id}`  ")
    lines.append(f"**Theme:** {puzzle.theme or 'N/A'}  ")
    lines.append(f"**Strategy mix:** {puzzle.source_strategy}  ")
    lines.append(f"**Validator status:** {report.status}  ")
    lines.append(f"**Quality score:** {report.quality_score:.2f}  ")
    lines.append(f"**Blind solver status:** {solver.status}  ")
    lines.append(f"**Nearest-reference similarity:** {float(nearest.get('similarity', 0.0)):.3f}  ")
    lines.append("")
    lines.append("**Displayed words**")
    lines.append(", ".join(puzzle.words))
    lines.append("")
    lines.append("**Answer groups**")
    for group in puzzle.groups:
        lines.append(f"- **{group.category}** ({group.strategy}, {group.difficulty}): {', '.join(group.words)}")
    if item["issues"]:
        lines.append("")
        lines.append("**Validation issues**")
        for issue in item["issues"]:
            lines.append(f"- {issue.severity.upper()} `{issue.code}`: {issue.message}")
    return "\n".join(lines)

strong_item = max(results, key=lambda item: item["report"].quality_score)
flagged_candidates = [item for item in results if item["report"].status != "publish" or item["solver"].status != "unique_match"]
flagged_item = flagged_candidates[0] if flagged_candidates else min(results, key=lambda item: item["report"].quality_score)

for label, item in [("Strong example", strong_item), ("Flagged example", flagged_item)]:
    text = f"## {label}\n\n" + board_markdown(item)
    if Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


## What this notebook demonstrates

This notebook is intentionally compact. It does not replace the website.
and instead, it shows that the backend generator-and-audit path is local, reproducible, and inspectable.